In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# 设定随机数种子
np.random.seed(123)

# 创建数据
n = 80 # 样本量，可修改
x = np.linspace(-3, 3, n).reshape(-1, 1)
t = 0.5 * x.ravel()**2 + x.ravel() # 真实的二次关系
y = t + np.random.randn(n) * 0.6 # 添加噪声

def plot_polynomial_fit(x, t, y, deg):
    """
    对给定数据进行多项式拟合，并绘制出原始数据、拟合结果和理想结果的图像。

    参数
    ----------
    x : ndarray
        x坐标的数据
    y : ndarray
        y坐标的数据
    deg : int
        多项式的阶数
    np.polyfit(x, y, deg) :
        对数据点 (x, y) 进行 最小二乘多项式拟合，返回多项式系数
    np.poly1d(...) :
        通过多项式系数构造多项式函数
    """
    # 将x展平为一维，供 np.polyfit 使用
    x_flat = x.ravel()
    # 对数据进行多项式拟合
    p = np.poly1d(np.polyfit(x_flat, y, deg))


    # 绘制原始数据（红色圆点）、拟合结果（蓝色实线）和理想结果（红色虚线）
    plt.plot(x, y, 'ro', label='Original Data')
    plt.plot(x, p(x), '-', label=f'Degree {deg} Fit')
    plt.plot(x, 0.5 * x_flat**2 + x_flat, 'r--', label='Ideal Result')

    # 显示图例
    plt.legend()

plt.figure(figsize=(18, 4), dpi=200)
degrees = [1, 3, 10]  # 多项式的阶数
titles = ['Under Fitting', 'Fitting', 'Over Fitting']  # 图像的标题
for index, deg in enumerate(degrees):
    plt.subplot(1, 3, index + 1)
    plot_polynomial_fit(x, t, y, deg)
    plt.title(titles[index], fontsize=20)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error


# 正则化限制模型的复杂度，以前面degree=10作为过拟合对象进行正则化研究
degree = 10
# PolynomialFeatures(include_bias=False) 避免与线性模型截距重复
models = {
    'No regularization': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        LinearRegression()
    ),
    'L1 (Lasso)': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        Lasso(alpha=0.01, max_iter=100000)
    ),
    'L2 (Ridge)': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        Ridge(alpha=0.1)
    ),
    'ElasticNet': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=100000)
    )
}

plt. figure(figsize=(10, 6))  # 新建一个图
# 各正则化模型预测曲线
for name, model in models.items():
    model.fit(x, y)

    y_pred = model.predict(x)

    # 训练误差和测试误差
    train_mse = mean_squared_error(y, y_pred)
    test_mse = mean_squared_error(t, y_pred)

    plt.plot(x, y_pred, '--', linewidth=2,
             label=f'{name} (train_MSE: {train_mse:.3f}, test_MSE: {test_mse:.3f})')
# 理想结果曲线
plt.plot(x, 0.5 * x.ravel()**2 + x.ravel(), 'k-', linewidth=2, label='Ideal Result')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Comparison of Different Regularization Methods under Overfitting (Degree=10)')
plt.legend(loc='best', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== Dropout & Early Stopping ====================
# dropout 和早停是神经网络（深度学习）中的正则化方法，无法直接放进
# 上面的 sklearn Pipeline，因此这里用 Keras 构建一个容量较大、容易
# 过拟合的 MLP，来演示两种方法的正则化效果。
import matplotlib.pyplot as plt  # 绘图库，用于最后画预测曲线
import tensorflow as tf  # 深度学习框架（Colab 已预装）
from tensorflow.keras.models import Sequential  # 顺序模型：网络层按顺序一层接一层
from tensorflow.keras.layers import Dense, Dropout  # Dense=全连接层，Dropout=随机丢弃神经元
from tensorflow.keras.callbacks import EarlyStopping  # 早停回调：验证损失不再下降时提前停止训练
from sklearn.preprocessing import StandardScaler  # 数据标准化工具
from sklearn.metrics import mean_squared_error  # 均方误差，用于评估预测效果

# 神经网络对输入尺度敏感，先将 x 标准化（变为均值为0、标准差为1），帮助训练更快收敛
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)  # 先用训练数据算出均值/标准差，再对 x 做标准化
y_2d = y.reshape(-1, 1)  # Keras 要求目标值 y 是二维形状 (样本数, 1)，所以从一维变成二维


def build_mlp(with_dropout, dropout_rate=0.3, seed=0):
    """构建一个容量较大的 MLP；with_dropout=True 时在隐藏层之间加入 Dropout 层"""
    tf.keras.utils.set_random_seed(seed)  # 固定随机种子，保证每次初始权重相同，便于公平对比
    model = Sequential()  # 创建顺序模型（网络层按添加顺序依次连接）
    model.add(Dense(128, activation='relu', input_shape=(1,)))  # 第1层：128个神经元，relu激活，输入是1个特征(x)
    if with_dropout:
        model.add(Dropout(dropout_rate))  # 训练时随机丢弃30%的神经元，防止网络过度依赖某些神经元
    model.add(Dense(128, activation='relu'))  # 第2层：128个神经元
    if with_dropout:
        model.add(Dropout(dropout_rate))
    model.add(Dense(1))  # 输出层：1个神经元，输出预测的 y 值（回归问题）
    model.compile(optimizer='adam', loss='mse')  # 优化器用 adam，损失函数用均方误差 mse
    return model


# 训练四个结构完全相同的模型，形成 2×2 对比（是否 Dropout × 是否早停）：
#   1) 无 Dropout + 早停
#   2) Dropout(0.3) + 早停
#   3) Dropout(0.3) + 不使用早停
#   4) 无 Dropout + 不使用早停（跑满全部 200 轮），用于对比
epochs_dict = {}  # 用于保存每个模型的实际训练轮数
results = {}  # 字典：模型名 -> (训练好的模型, 训练历史, 是否使用早停)
for name, use_dropout, use_early_stop in [
        ('No Dropout, Early Stopping', False, True),
        ('Dropout (0.3), Early Stopping', True, True),
        ('No Dropout, No Early Stopping', False, False),
        ('Dropout (0.3), No Early Stopping', True, False)]:
    model = build_mlp(with_dropout=use_dropout)  # 按配置构建模型
    callbacks = []  # 回调列表，先为空
    if use_early_stop:
        # 监测验证集损失，连续 50 轮不下降即停止并恢复最优权重
        callbacks.append(EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True))
    # 训练模型：最多 200 轮；validation_split=0.2 表示拿出 20% 数据作为验证集；
    # batch_size=16 表示每批用 16 个样本更新一次权重；verbose=0 表示不打印训练过程
    history = model.fit(x_scaled, y_2d, epochs=200, validation_split=0.2,
                        batch_size=16, callbacks=callbacks, verbose=0)
    epochs_dict[name] = len(history.history["loss"])
    results[name] = (model, history, use_early_stop)


# 预测曲线对比（沿用上面的 train/test MSE 度量方式；四个模型两两对比 Dropout 和早停的效果）
plt.figure(figsize=(10, 6))  # 新建一张 10×6 英寸的图
for name, (model, _, _) in results.items():  # 遍历四个模型
    y_pred = model.predict(scaler.transform(x), verbose=0).ravel()  # 预测：x 先标准化再输入网络，输出转为一维
    train_mse = mean_squared_error(y, y_pred)  # 训练误差：预测值 vs 带噪声的真实样本 y
    test_mse = mean_squared_error(t, y_pred)  # 测试误差：预测值 vs 无噪声的理想曲线 t
    plt.plot(x, y_pred, '--', linewidth=2,  # 画预测曲线
             label=f'{name}: trained {epochs_dict.get(name)} epchos (train_MSE: {train_mse:.3f}, test_MSE: {test_mse:.3f})')
plt.plot(x, t, 'k-', linewidth=2, label='Ideal Result')  # 理想曲线（无噪声）作为参照
plt.xlabel('x')
plt.ylabel('y')
plt.title('Dropout & Early Stopping Regularization (MLP)')
plt.legend(loc='best', fontsize=9)  # 图例
plt.grid(True, alpha=0.3)  # 网格线
plt.tight_layout()  # 自动调整布局
plt.show()  # 显示图像